<a href="https://colab.research.google.com/github/CpfPatrick/aquarium-yolo11-portfolio/blob/main/notebooks/01_trained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Runtime and GPU check

In [4]:
import platform
import torch

print(platform.python_version())
print(torch.__version__)
print(torch.cuda.is_available())

3.12.13
2.11.0+cu128
True


In [5]:
gpu_name = torch.cuda.get_device_name(0)
properties = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)

print(gpu_name)
print(torch.cuda.get_device_capability(0))
print(total_bytes/1024**3)
print(torch.cuda.memory_allocated(0))
print(f"Allocated:  {torch.cuda.memory_allocated(0) / 1024**3:.2f} GiB")
print(f"Reserved:   {torch.cuda.memory_reserved(0) / 1024**3:.2f} GiB")


Tesla T4
(7, 5)
14.56317138671875
0
Allocated:  0.00 GiB
Reserved:   0.00 GiB


In [6]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/fishsense-object-detection"
)
DATA_DIR = DRIVE_ROOT / "data"
RUNS_DIR = DRIVE_ROOT / "runs"
WEIGHTS_DIR = DRIVE_ROOT / "weights"
REPORTS_DIR = DRIVE_ROOT / "reports"

In [8]:
for directory in [
    DRIVE_ROOT,
    DATA_DIR,
    WEIGHTS_DIR,
    REPORTS_DIR
]: directory.mkdir(parents = True, exist_ok = True)


print("Project root:", DRIVE_ROOT)
print("Data:", DATA_DIR)
print("Runs:", RUNS_DIR)
print("Weights:", WEIGHTS_DIR)
print("Reports:", REPORTS_DIR)

Project root: /content/drive/MyDrive/fishsense-object-detection
Data: /content/drive/MyDrive/fishsense-object-detection/data
Runs: /content/drive/MyDrive/fishsense-object-detection/runs
Weights: /content/drive/MyDrive/fishsense-object-detection/weights
Reports: /content/drive/MyDrive/fishsense-object-detection/reports


In [9]:
%pip install "ultralytics==8.4.115" "jedi==0.19.2"
%pip check

No broken requirements found.


Smoke Test

In [15]:
import torch

print(torch.cuda.is_available())

True


In [21]:
import json
from ultralytics import YOLO

MODEL_PATH = WEIGHTS_DIR / "yolo11n.pt"
SMOKE_DIR = REPORTS_DIR / "smoke"
OUTPUT_PATH = SMOKE_DIR / "bus_yolo11n_gpu.jpg"
MANIFEST_PATH = SMOKE_DIR / "smoke_manifest.json"

SAMPLE_URL = "https://ultralytics.com/images/bus.jpg"

WEIGHTS_DIR.mkdir(parents = True, exist_ok = True)
SMOKE_DIR.mkdir(parents = True, exist_ok = True)

model = YOLO(str(MODEL_PATH))

result = model.predict(
    source = SAMPLE_URL,
    device = 0,
    imgsz = 640,
    verbose = True
)

# organize result:

inference_device = result[0].boxes.data.device
result[0].save(filename = str(OUTPUT_PATH))
manifest = {
    "model": "yolo11n.pt",
    "model_path": str(MODEL_PATH),
    "source": SAMPLE_URL,
    "gpu": gpu_name,
    "inference_device": str(inference_device),
    "detections": len(result[0].boxes),
    "output_path": str(OUTPUT_PATH),
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Prediction saved:", OUTPUT_PATH)


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 /content/bus.jpg: 640x480 4 persons, 1 bus, 7.3ms
Speed: 2.4ms preprocess, 7.3ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 480)
{
  "model": "yolo11n.pt",
  "model_path": "/content/drive/MyDrive/fishsense-object-detection/weights/yolo11n.pt",
  "source": "https://ultralytics.com/images/bus.jpg",
  "gpu": "Tesla T4",
  "inference_device": "cuda:0",
  "detections": 5,
  "output_path": "/content/drive/MyDrive/fishsense-object-detection/reports/smoke/bus_yolo11n_gpu.jpg"
}
Prediction saved: /content/drive/MyDrive/fishsense-object-detection/reports/smoke/bus_yolo11n_gpu.jpg
